# Model Architecture Visualization

Ten notebook służy do wizualizacji i analizy struktury modeli używanych w projekcie StatsBomb Scout.

Dostępne modele:
- **LSTM**: Podstawowy model LSTM z dwiema warstwami
- **Attention LSTM**: LSTM z mechanizmem uwagi (attention)
- **BiGRU**: Bidirectional GRU z temporal attention pooling
- **Transformer**: Model oparty na architekturze Transformer

In [14]:
import sys
sys.path.append('src')

from src.ml.models.lstm import LSTMSequenceModel
from src.ml.models.attention_lstm import AttentionLSTMModel
from src.ml.models.bigru import build_seq_value_model
from src.ml.models.transformer import TransformerSequenceModel

from tensorflow import keras
import tensorflow as tf

# Parametry wejściowe (dostosuj według potrzeb)
INPUT_SHAPE = (6, 46)  # (max_sequence_length, num_features)

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.20.0


## 1. LSTM Model

Podstawowy model LSTM składający się z:
- Warstwy maskującej (Masking) - ignoruje padding
- Dwóch warstw LSTM (128 jednostek, 64 jednostki)
- Warstw Dropout dla regularyzacji
- Warstwy Dense (64 jednostki) z aktywacją ReLU
- Wyjściowej warstwy Dense (1 jednostka) - regresja

In [15]:
# LSTM Model
lstm_model = LSTMSequenceModel(
    input_shape=INPUT_SHAPE,
    lstm_units=128,
    dropout=0.3
)
lstm_model.build()

print("=" * 80)
print("LSTM MODEL SUMMARY")
print("=" * 80)
lstm_model.model.summary()

LSTM MODEL SUMMARY


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 6, 46)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, 6, 46)     │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_4 (Masking) │ (None, 6, 46)     │          0 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_4 (Any)         │ (None, 6)         │          0 │ not_equal_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ (None, 6, 128)    │     89,600 │ masking_4[0][0],  │
│                     │                   │            │ any_4[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 6, 128)    │          0 │ lstm_4[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ (None, 64)        │     49,408 │ dropout_14[0][0], │
│                     │                   │            │ any_4[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 64)        │          0 │ lstm_5[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 1)         │         65 │ dropout_15[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 139,073 (543.25 KB)

 Trainable params: 139,073 (543.25 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# Wizualizacja graficzna LSTM
keras.utils.plot_model(
    lstm_model.model,
    to_file='models/lstm_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=False,
    dpi=96
)
print("Diagram zapisany jako: models/lstm_architecture.png")

Diagram zapisany jako: models/lstm_architecture.png


## 2. Attention LSTM Model

LSTM z mechanizmem uwagi:
- Warstwy maskującej (Masking)
- Bidirectional LSTM (2 × 128 = 256 jednostek)
- Drugiej warstwy LSTM (128 jednostek)
- **Custom AttentionLayer** - oblicza wagi uwagi dla każdego kroku czasowego
- Warstwy Dense (64 jednostki)
- Dwa wyjścia: wartość predykcji oraz wagi uwagi

In [17]:
# Attention LSTM Model
attention_lstm_model = AttentionLSTMModel(
    input_shape=INPUT_SHAPE,
    lstm_units=128,
    dropout=0.3
)
attention_lstm_model.build()

print("=" * 80)
print("ATTENTION LSTM MODEL SUMMARY")
print("=" * 80)
attention_lstm_model.model.summary()

ATTENTION LSTM MODEL SUMMARY


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 6, 46)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_5         │ (None, 6, 46)     │          0 │ sequence_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_5 (Masking) │ (None, 6, 46)     │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_5 (Any)         │ (None, 6)         │          0 │ not_equal_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 6, 256)    │    179,200 │ masking_5[0][0],  │
│ (Bidirectional)     │                   │            │ any_5[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 6, 256)    │          0 │ bidirectional_1[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ (None, 6, 128)    │    197,120 │ dropout_16[0][0], │
│                     │                   │            │ any_5[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 6, 128)    │          0 │ lstm_7[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_weights   │ [(None, 128),     │        134 │ dropout_17[0][0], │
│ (AttentionLayer)    │ (None, 6)]        │            │ any_5[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 64)        │      8,256 │ attention_weight… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 64)        │          0 │ dense_12[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ value (Dense)       │ (None, 1)         │         65 │ dropout_18[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 384,775 (1.47 MB)

 Trainable params: 384,775 (1.47 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
# Wizualizacja graficzna Attention LSTM
keras.utils.plot_model(
    attention_lstm_model.model,
    to_file='models/attention_lstm_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=False,
    dpi=96
)
print("Diagram zapisany jako: models/attention_lstm_architecture.png")

Diagram zapisany jako: models/attention_lstm_architecture.png


## 3. BiGRU Model

Bidirectional GRU z temporal attention:
- Warstwy maskującej (Masking)
- Bidirectional GRU (2 × 128 = 256 jednostek) z regularyzacją L2 i constraints
- Layer Normalization
- **TemporalAttentionPooling** - aggreguje sekwencję z wagami uwagi
- Dense (128 jednostek) z Batch Normalization
- Wyjście: wartość predykcji i wagi uwagi (opcjonalnie)

In [19]:
# BiGRU Model
bigru_model = build_seq_value_model(
    input_shape=INPUT_SHAPE,
    rnn_units=128,
    attn_hidden=64,
    dropout=0.2,
    recurrent_dropout=0.15,
    l2_reg=0.01,
    return_attention=True  # Zwróć również wagi uwagi
)

print("=" * 80)
print("BiGRU MODEL SUMMARY")
print("=" * 80)
bigru_model.summary()

BiGRU MODEL SUMMARY


Model: "bigru_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 6, 46)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_6         │ (None, 6, 46)     │          0 │ sequence_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_6 (Masking) │ (None, 6, 46)     │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_6 (Any)         │ (None, 6)         │          0 │ not_equal_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bigru               │ (None, 6, 256)    │    135,168 │ masking_6[0][0],  │
│ (Bidirectional)     │                   │            │ any_6[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 6, 256)    │        512 │ bigru[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_pool           │ [(None, 256),     │     16,513 │ layer_normalizat… │
│ (TemporalAttention… │ (None, 6)]        │            │ any_6[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 128)       │     32,896 │ attn_pool[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_15[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_20          │ (None, 128)       │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_weights   │ (None, 6)         │          0 │ attn_pool[0][1]   │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ value (Dense)       │ (None, 1)         │        129 │ dropout_20[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 185,730 (725.51 KB)

 Trainable params: 185,474 (724.51 KB)

 Non-trainable params: 256 (1.00 KB)

In [20]:
# Wizualizacja graficzna BiGRU
keras.utils.plot_model(
    bigru_model,
    to_file='models/bigru_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=False,
    dpi=96
)
print("Diagram zapisany jako: models/bigru_architecture.png")

Diagram zapisany jako: models/bigru_architecture.png


## 4. Transformer Model

Model oparty na architekturze Transformer:
- Warstwy maskującej (Masking)
- Projekcja do d_model wymiarów
- **Positional Encoding** - dodaje informację o pozycji w sekwencji
- **Transformer Encoder Blocks** (domyślnie 2):
  - Multi-Head Self-Attention (4 głowice)
  - Feed-Forward Network (512 jednostek)
  - Layer Normalization i residual connections
- Global Average Pooling - agregacja sekwencji
- Dense (64 jednostki)
- Dwa wyjścia: wartość predykcji oraz wagi uwagi

In [ ]:
# Transformer Model
transformer_model = TransformerSequenceModel(
    input_shape=INPUT_SHAPE,
    num_heads=4,
    d_model=128,
    ff_dim=512,
    num_blocks=2,
    dropout=0.1
)
transformer_model.build()

print("=" * 80)
print("TRANSFORMER MODEL SUMMARY")
print("=" * 80)
transformer_model.model.summary()

In [22]:
# Wizualizacja graficzna Transformer
keras.utils.plot_model(
    transformer_model.model,
    to_file='models/transformer_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=False,
    dpi=96
)
print("Diagram zapisany jako: models/transformer_architecture.png")

Diagram zapisany jako: models/transformer_architecture.png


## 5. Porównanie liczby parametrów

Zestawienie wszystkich modeli z liczbą parametrów trenowalnych.

In [23]:
import pandas as pd

# Zbierz statystyki wszystkich modeli
models_comparison = [
    {
        'Model': 'LSTM',
        'Total Parameters': lstm_model.model.count_params(),
        'Trainable Parameters': sum([tf.size(w).numpy() for w in lstm_model.model.trainable_weights]),
        'Architecture': 'LSTM → LSTM → Dense'
    },
    {
        'Model': 'Attention LSTM',
        'Total Parameters': attention_lstm_model.model.count_params(),
        'Trainable Parameters': sum([tf.size(w).numpy() for w in attention_lstm_model.model.trainable_weights]),
        'Architecture': 'Bi-LSTM → LSTM → Attention → Dense'
    },
    {
        'Model': 'BiGRU',
        'Total Parameters': bigru_model.count_params(),
        'Trainable Parameters': sum([tf.size(w).numpy() for w in bigru_model.trainable_weights]),
        'Architecture': 'Bi-GRU → Temporal Attention → Dense'
    },
    {
        'Model': 'Transformer',
        'Total Parameters': transformer_model.model.count_params(),
        'Trainable Parameters': sum([tf.size(w).numpy() for w in transformer_model.model.trainable_weights]),
        'Architecture': 'Positional Encoding → Transformer Blocks → Pooling → Dense'
    }
]

df_comparison = pd.DataFrame(models_comparison)
df_comparison['Total Parameters'] = df_comparison['Total Parameters'].apply(lambda x: f"{x:,}")
df_comparison['Trainable Parameters'] = df_comparison['Trainable Parameters'].apply(lambda x: f"{x:,}")

print("\n" + "=" * 100)
print("MODEL COMPARISON")
print("=" * 100)
print(df_comparison.to_string(index=False))
print("=" * 100)


MODEL COMPARISON
         Model Total Parameters Trainable Parameters                                               Architecture
          LSTM          139,073              139,073                                        LSTM → LSTM → Dense
Attention LSTM          384,775              384,775                         Bi-LSTM → LSTM → Attention → Dense
         BiGRU          185,730              185,474                        Bi-GRU → Temporal Attention → Dense
   Transformer          411,649              411,649 Positional Encoding → Transformer Blocks → Pooling → Dense


## 6. Analiza poszczególnych warstw

Szczegółowa analiza konkretnych warstw w wybranym modelu.

In [24]:
# Przykład: Analiza warstw Attention LSTM
print("\nDetailed Layer Analysis - Attention LSTM Model:")
print("=" * 80)

for i, layer in enumerate(attention_lstm_model.model.layers):
    print(f"\nLayer {i}: {layer.name}")
    print(f"  Type: {type(layer).__name__}")

    # Obsługa InputLayer i innych warstw bez output_shape
    if hasattr(layer, 'output_shape'):
        print(f"  Output Shape: {layer.output_shape}")
    elif hasattr(layer, 'output'):
        # Sprawdź czy output to lista (warstwy z wieloma wyjściami)
        if isinstance(layer.output, list):
            print(f"  Output Shapes: {[out.shape for out in layer.output]}")
        else:
            print(f"  Output Shape: {layer.output.shape}")

    if hasattr(layer, 'units'):
        print(f"  Units: {layer.units}")
    if hasattr(layer, 'activation'):
        print(f"  Activation: {layer.activation}")
    
    # Liczba parametrów w warstwie
    trainable_params = sum([tf.size(w).numpy() for w in layer.trainable_weights])
    non_trainable_params = sum([tf.size(w).numpy() for w in layer.non_trainable_weights])
    
    if trainable_params > 0 or non_trainable_params > 0:
        print(f"  Trainable params: {trainable_params:,}")
        print(f"  Non-trainable params: {non_trainable_params:,}")


Detailed Layer Analysis - Attention LSTM Model:

Layer 0: sequence_input
  Type: InputLayer
  Output Shape: (None, 6, 46)

Layer 1: masking_5
  Type: Masking
  Output Shape: (None, 6, 46)

Layer 2: bidirectional_1
  Type: Bidirectional
  Output Shape: (None, 6, 256)
  Trainable params: 179,200
  Non-trainable params: 0

Layer 3: dropout_16
  Type: Dropout
  Output Shape: (None, 6, 256)

Layer 4: lstm_7
  Type: LSTM
  Output Shape: (None, 6, 128)
  Units: 128
  Activation: <function tanh at 0x152fa4ae0>
  Trainable params: 197,120
  Non-trainable params: 0

Layer 5: dropout_17
  Type: Dropout
  Output Shape: (None, 6, 128)

Layer 6: attention_weights
  Type: AttentionLayer
  Output Shapes: [(None, 128), (None, 6)]
  Trainable params: 134
  Non-trainable params: 0

Layer 7: dense_12
  Type: Dense
  Output Shape: (None, 64)
  Units: 64
  Activation: <function relu at 0x152b0e840>
  Trainable params: 8,256
  Non-trainable params: 0

Layer 8: dropout_18
  Type: Dropout
  Output Shape: (Non

## 7. Testowe przewidywanie

Sprawdzenie czy model działa poprawnie na przykładowych danych.

In [25]:
import numpy as np

# Wygeneruj przykładowe dane
batch_size = 2
sample_input = np.random.randn(batch_size, INPUT_SHAPE[0], INPUT_SHAPE[1]).astype(np.float32)

# Dodaj trochę padding'u (zera na końcu)
sample_input[:, -5:, :] = 0.0

print("\nTest Prediction:")
print("=" * 80)
print(f"Input shape: {sample_input.shape}")

# LSTM (pojedyncze wyjście)
lstm_pred = lstm_model.model.predict(sample_input, verbose=0)
print(f"\nLSTM prediction shape: {lstm_pred.shape}")
print(f"LSTM prediction values: {lstm_pred.flatten()}")

# Attention LSTM (dwa wyjścia: value + attention_weights)
attention_lstm_pred = attention_lstm_model.model.predict(sample_input, verbose=0)
print(f"\nAttention LSTM prediction:")
print(f"  Value shape: {attention_lstm_pred['value'].shape}")
print(f"  Value: {attention_lstm_pred['value'].flatten()}")
print(f"  Attention weights shape: {attention_lstm_pred['attention_weights'].shape}")
print(f"  Attention weights sum (should be ~1.0): {attention_lstm_pred['attention_weights'].sum(axis=1)}")


Test Prediction:
Input shape: (2, 6, 46)

LSTM prediction shape: (2, 1)
LSTM prediction values: [0.03735787 0.01113761]

Attention LSTM prediction:
  Value shape: (2, 1)
  Value: [ 0.00747545 -0.01727388]
  Attention weights shape: (2, 6)
  Attention weights sum (should be ~1.0): [1. 1.]


## 8. Eksport do LaTeX/dokumentacji

Przygotowanie tabel do publikacji naukowej.

In [26]:
# Eksport do LaTeX
latex_table = df_comparison.to_latex(index=False, caption="Porównanie architektur modeli", label="tab:model_comparison")
print("\nLaTeX Table:")
print(latex_table)

# Zapisz do pliku
with open('models/model_comparison_table.tex', 'w') as f:
    f.write(latex_table)
print("\nTabela LaTeX zapisana jako: models/model_comparison_table.tex")


LaTeX Table:
\begin{table}
\caption{Porównanie architektur modeli}
\label{tab:model_comparison}
\begin{tabular}{llll}
\toprule
Model & Total Parameters & Trainable Parameters & Architecture \\
\midrule
LSTM & 139,073 & 139,073 & LSTM → LSTM → Dense \\
Attention LSTM & 384,775 & 384,775 & Bi-LSTM → LSTM → Attention → Dense \\
BiGRU & 185,730 & 185,474 & Bi-GRU → Temporal Attention → Dense \\
Transformer & 411,649 & 411,649 & Positional Encoding → Transformer Blocks → Pooling → Dense \\
\bottomrule
\end{tabular}
\end{table}


Tabela LaTeX zapisana jako: models/model_comparison_table.tex
